### Load packages

In [1]:
import cloudpickle as pickle
from datetime import datetime
import jax
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

from utils import *
from model_definition.network_model_utils import *
from tvboptim.types import Parameter, BoundedParameter

jax.config.update("jax_enable_x64", True)

Cache stored here: c:\Users\Bruna\Documents\Cogmaster\M2\Stage\scripts\TVBOptim\cache\ei_tuning


### Set repositories and global parameters

In [5]:
# Set directory information
data_dir = "./"
cond0_filename = "TS_Control.npy"
cond1_filename = "TS_Schizo.npy"
result_dir = r"D:\Cogmaster\M2\stage\results"

# Set dataset parameters
n_sub = 48
n_nodes = 68 # size of network for AAL90
conds = ['CTR', 'SCZ']
n_cond = len(conds) # number of conditions

# Simulation parameters
t1 = 314_000   # Simulation duration (ms) matching empirical data (=304_000) + transient time (~10_000 ms)
dt = 4.0      # Integration timestep (ms) matching original script
bold_TR = 2000.0 # BOLD sampling period (ms)
transient_lim = 5 # Number of time points to remove as transient (transient_lim * dt ms)
target_fic = 0.25  # FIC tuning parameter: Target excitatory activity level

# Gradient descent parameters
learning_rate = 0.0325
max_steps = 120

# Other parameters
n_tau = 2 # number of lags for lagged FC

### Define functions

In [ ]:
def load_and_organize_bold(data_dir: str | None = None, 
                           cond0_filename: str | None = None, 
                           cond1_filename: str | None = None,
                           n_sub: int = 48, 
                           n_nodes: int = 68
                           ) -> np.ndarray:
    """
    Load and organize BOLD signal data for multiple subjects.
    
    Parameters
    ----------
    data_dir (str): Directory containing the BOLD signal files.
    cond0_filename (str): Filename for the control group BOLD signal.
    cond1_filename (str): Filename for the schizophrenic group BOLD signal.
    n_sub (int): Number of subjects.
    n_nodes (int): Number of nodes or regions.

    Returns
    -------
    np.ndarray: Organized BOLD signal data of shape (n_sub, n_time_points, n_regions, n_cond).
    """
    if data_dir is None or cond0_filename is None or cond1_filename is None:
        raise ValueError("data_dir, cond0_filename, and cond1_filename must be provided.")

    ## Load time-series bold data from two conditions, in this case, schizophrenic and control groups
    TS_CTR  = np.load(os.path.join(data_dir, cond0_filename))
    TS_SCZ  = np.load(os.path.join(data_dir, cond1_filename))

    ## Organize the data
    # Separate the participants by condition
    condition_0 = TS_CTR[0:n_sub, 0:n_nodes, :]  
    condition_1 = TS_SCZ[0:n_sub, 0:n_nodes, :]  

    # Determine the maximum number of participants in either condition (for alignment)
    max_participants = max(condition_0.shape[2], condition_1.shape[2])

    # Pad the smaller group to match the size of the larger one along the participant dimension
    condition_0_padded = np.pad(condition_0, ((0, 0), (0, 0), (0, max_participants - condition_0.shape[2])), mode='constant')
    condition_1_padded = np.pad(condition_1, ((0, 0), (0, 0), (0, max_participants - condition_1.shape[2])), mode='constant')

    # Stack the conditions along the fourth dimension
    new_array = np.stack((condition_0_padded, condition_1_padded), axis=3)
    
    return new_array

def load_structural_connectivity(sc_filepath: str | None = None,
                                 tl_filepath: str | None = None,
                                 centers_filepath: str | None = None) -> tuple[np.ndarray, pd.DataFrame, list]:
    """
    Load the structural connectivity matrix from a .npy file.
    
    Parameters
    ----------
    sc_filepath (str | None): Filepath for the structural connectivity matrix (.mat format expected).
    tl_filepath (str | None): Filepath for the tract lengths file.
    centers_filepath (str | None): Filepath for the region centers file.

    Returns
    -------
    tuple[np.ndarray, pd.DataFrame, list]: A tuple containing the normalized structural connectivity matrix, tract lengths DataFrame, and region labels.
    """
    if sc_filepath is None or tl_filepath is None or centers_filepath is None:
        raise ValueError("All filepaths (connectome, tract lengths, region centers) must be provided.")
    
    # Weights
    SCR = sio.loadmat(sc_filepath)['matrix']
    weights = SCR / np.max(SCR)

    # Delays
    lengths = pd.read_csv(tl_filepath)
    speed = 3.0
    delays = lengths / speed

    # Load region labels and coordinates
    df = pd.read_csv(
        centers_filepath,
        sep='\t',
        header=None,
        dtype={1: float, 2: float, 3: float},
        names=['label', 'x', 'y', 'z']
    )

    labels = df['label'].tolist()
    
    return weights, delays, labels


### Main script

In [3]:
new_array = load_and_organize_bold(data_dir = data_dir, cond0_filename = cond0_filename, cond1_filename = cond1_filename,
                                   n_sub = n_sub, n_nodes = n_nodes)

In [4]:
Q0_emp_all = np.zeros((n_sub, n_nodes, n_nodes, n_cond))  # shape: (n_sub, n_tau, n_nodes, n_nodes, n_cond)
Q1_emp_all = np.zeros((n_sub, n_nodes, n_nodes, n_cond))

for participant_idx in range(n_sub):
    for condition_idx in range(n_cond):
        # Get empirical time series of interest 
        ts = new_array[participant_idx,:,:,condition_idx]
        # Take the transpose for the lagged FC matrices computation
        X_emp = ts.T
    
        # Z-score the empirical time series per region
        z_scored_emp = z_score_per_region(X_emp)

        # Compute empirical lagged FC matrices
        Q_emp_single = lagged_fc_matrices(z_scored_emp, n_tau=n_tau, diag_zero=True)
        Q0_emp_single = Q_emp_single[0]  # FC0 (zero-lag)
        Q1_emp_single = Q_emp_single[1]  # FC1 (lag-1)

        Q0_emp_all[participant_idx, :, :, condition_idx] = Q0_emp_single
        Q1_emp_all[participant_idx, :, :, condition_idx] = Q1_emp_single
    
print("Empirical time series shape (time points x regions):", X_emp.shape)
print("Empirical FC0 shape (regions x regions):", Q0_emp_all.shape)
print("Empirical FC1 shape (regions x regions):", Q1_emp_all.shape)

Empirical time series shape (time points x regions): (152, 68)
Empirical FC0 shape (regions x regions): (48, 68, 68, 2)
Empirical FC1 shape (regions x regions): (48, 68, 68, 2)


In [6]:
Q0_save_path = os.path.join(result_dir, "Q0_emp_all.npy")
Q1_save_path =  os.path.join(result_dir, "Q1_emp_all.npy")
np.save(Q0_save_path, Q0_emp_all)
np.save(Q1_save_path, Q1_emp_all)

In [ ]:
subj_plot_index = 0  # Index for the participant and condition to plot
cond_plot_index = 0  # Index for the condition to plot

Q0_emp_plot = Q0_emp_all[subj_plot_index, :, :, cond_plot_index]  # FC0 for the selected participant and condition
Q1_emp_plot = Q1_emp_all[subj_plot_index, :, :, cond_plot_index]  # Q1 for the selected participant and condition

# Plot Q 0 and Q 1 for both empirical and simulated data
fig, axes = plt.subplots(1,2, figsize=(12, 10))
# FC0 - Empirical
im0 = axes[0].imshow(Q0_emp_plot, vmin=-1, vmax=1, cmap='cividis')
axes[0].set_title('Empirical FC0 (Lag 0)')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
# FC1 - Empirical
im2 = axes[1].imshow(Q1_emp_plot, vmin=-1, vmax=1, cmap='cividis')
axes[1].set_title('Empirical FC1 (Lag 1)')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
plt.suptitle(f'Participant {subj_plot_index}, Condition {conds[cond_plot_index]}', fontsize=16, y=0.739)
plt.show()

In [ ]:
sc_path = 'SC_EnigmadK68.mat'
tl_path = 'tract_lengths.csv'
centers_path = 'centers.txt'

weights, delays, labels = load_structural_connectivity(sc_filepath=sc_path, tl_filepath=tl_path, centers_filepath=centers_path)

In [ ]:
# Build a single network model using the structural connectivity and region labels
network = build_network_model(weights=weights, labels=labels)